# LeetCode 3359 — Find Sorted Submatrices With Maximum Element at Most K bằng Python

- Link: https://leetcode.com/problems/find-sorted-submatrices-with-maximum-element-at-most-k/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Đếm hình chữ nhật con** kiểu "histogram": độ rộng hợp lệ kết thúc tại mỗi ô + **ngăn xếp đơn điệu** theo từng cột

> **Ghi chú Premium:** bài này là Premium nên LeetCode không trả về đề qua API công khai. Đề, ví dụ, ràng buộc lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3359/); trang không hiển thị mã nên **chữ ký `countSubmatrices(self, grid, k)` là suy đoán** — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (suy đoán):

```python
class Solution:
    def countSubmatrices(self, grid: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Đưa hai điều kiện (mọi phần tử `≤ k`, mỗi hàng **không tăng**) về một đại lượng cho từng ô: `W[i][j]` = độ rộng lớn nhất của đoạn hàng **kết thúc tại cột `j`** thỏa cả hai điều kiện.
2. Một hình chữ nhật con có góc dưới-phải `(i, j)` hợp lệ ⇔ độ rộng của nó ≤ `min W` trên các hàng của nó.
3. Đếm số hình chữ nhật có góc dưới-phải `(i, j)`: `Σ_{h} min(W[i−h+1..i][j])` — bằng **ngăn xếp đơn điệu** khi duyệt hàng xuống trong cùng một cột.
4. Tổng `O(m · n)`.
5. Kiểm chứng bằng vét cạn và bản `O(m² n)`.

## 2. Đề bài & ví dụ

Cho ma trận `grid` `m × n` và `k`. Đếm số ma trận con thỏa: phần tử lớn nhất `≤ k`, và **mỗi hàng** của ma trận con sắp **không tăng** (trái sang phải).

**Ví dụ 1:** `[[4,3,2,1],[8,7,6,1]], k = 3` → `8`. **Ví dụ 2:** ma trận `3 × 3` toàn 1, `k = 1` → `36`. **Ví dụ 3:** `[[1]], k = 1` → `1`.

**Ràng buộc (theo doocs):** `m, n ≤ 10^3`, giá trị ≤ `10^9`.

In [1]:
from typing import List

examples = [([[4, 3, 2, 1], [8, 7, 6, 1]], 3, 8), ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1, 36), ([[1]], 1, 1)]
for g, k, exp in examples:
    print(g, 'k =', k, '-> kỳ vọng', exp)

[[4, 3, 2, 1], [8, 7, 6, 1]] k = 3 -> kỳ vọng 8
[[1, 1, 1], [1, 1, 1], [1, 1, 1]] k = 1 -> kỳ vọng 36
[[1]] k = 1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Độ rộng hợp lệ kết thúc tại mỗi ô

`W[i][j] = 0` nếu `grid[i][j] > k`; ngược lại `W[i][j] = W[i][j−1] + 1` nếu `j > 0`, `W[i][j−1] > 0` và `grid[i][j−1] ≥ grid[i][j]` (vẫn không tăng), còn lại `1`.

In [2]:
def widths(g, k):
    m, n = len(g), len(g[0])
    W = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if g[i][j] <= k:
                if j and W[i][j - 1] and g[i][j - 1] >= g[i][j]:
                    W[i][j] = W[i][j - 1] + 1
                else:
                    W[i][j] = 1
    return W

print(widths(*examples[0][:2]))

[[0, 1, 2, 3], [0, 0, 0, 1]]


### 3.2. Đếm hình chữ nhật cố định góc dưới-phải

Với góc dưới-phải `(i, j)` và chiều cao `h`, số độ rộng hợp lệ là `min(W[i−h+1..i][j])` (mọi hàng phải hợp lệ trên đoạn cột đó). Tổng theo `h` cho số hình chữ nhật có góc dưới-phải `(i, j)`.

### 3.3. Ngăn xếp đơn điệu cho tổng các min hậu tố

Duyệt `i` tăng trong cột `j`, giữ ngăn xếp các cặp `(độ rộng, số hàng)` với độ rộng tăng dần từ đáy lên đỉnh, và `cur = Σ_h min(...)`. Thêm `W[i][j]`: gộp mọi khối có độ rộng `≥` nó (vì min của chúng giờ bị chặn bởi `W[i][j]`), cập nhật `cur`.

In [3]:
def sum_suffix_mins(arr):
    st, cur, out = [], 0, []
    for w in arr:
        cnt = 1
        while st and st[-1][0] >= w:
            pw, pc = st.pop(); cur -= pw * pc; cnt += pc
        st.append((w, cnt)); cur += w * cnt
        out.append(cur)
    return out

arr = [3, 1, 2, 2]
print(sum_suffix_mins(arr), [sum(min(arr[t:i + 1]) for t in range(i + 1)) for i in range(len(arr))])

[3, 2, 4, 6] [3, 2, 4, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`k = 3`): `W = [[0,1,2,3],[0,0,0,1]]`.

- Cột 3: hàng 0 có `W = 3` ⇒ 3 hình `1 × w`; hàng 1 có `W = 1` ⇒ hình `1 × 1` và hình `2 × 1` (cao 2, `min(3, 1) = 1`) ⇒ 2.
- Cột 2: hàng 0 `W = 2` ⇒ 2. Cột 1: hàng 0 `W = 1` ⇒ 1.

Tổng `3 + 2 + 2 + 1 = 8` ✓.

## 5. Cách 1 — Brute Force: mọi ma trận con

In [4]:
def count_brute(g, k):
    m, n = len(g), len(g[0]); c = 0
    for x1 in range(m):
        for x2 in range(x1, m):
            for y1 in range(n):
                for y2 in range(y1, n):
                    ok = all(g[x][y] <= k for x in range(x1, x2 + 1) for y in range(y1, y2 + 1))
                    ok = ok and all(g[x][y] >= g[x][y + 1] for x in range(x1, x2 + 1) for y in range(y1, y2))
                    c += ok
    return c

for g, k, exp in examples:
    print(count_brute(g, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
36 | kỳ vọng 36
1 | kỳ vọng 1


**Độ phức tạp:** `O(m² n² · m n)` — chỉ cho lưới rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `W` trong `O(mn)`.
2. Với mỗi cột `j`, duyệt `i = 0..m−1`, cập nhật ngăn xếp đơn điệu với `W[i][j]`, cộng `cur` (số hình có góc dưới-phải `(i, j)`) vào đáp án.

Mỗi phần tử vào/ra ngăn xếp một lần mỗi cột ⇒ `O(mn)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Với mỗi góc dưới-phải, đi lên và giữ min (`O(m² n)`)

In [5]:
def count_quadratic(g, k):
    W = widths(g, k); m, n = len(g), len(g[0]); c = 0
    for j in range(n):
        for i in range(m):
            mn = float('inf')
            for t in range(i, -1, -1):
                mn = min(mn, W[t][j])
                if mn == 0:
                    break
                c += mn
    return c

for g, k, exp in examples:
    print(count_quadratic(g, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
36 | kỳ vọng 36
1 | kỳ vọng 1


### 7.2. Cách 3 — Độ rộng + ngăn xếp đơn điệu theo cột (lời giải chính)

In [6]:
def count_stack(g, k):
    W = widths(g, k); m, n = len(g), len(g[0]); total = 0
    for j in range(n):
        total += sum(sum_suffix_mins([W[i][j] for i in range(m)]))
    return total

for g, k, exp in examples:
    print(count_stack(g, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
36 | kỳ vọng 36
1 | kỳ vọng 1


### 7.3. Cách 4 — Đổi vai: chiều cao theo cột, duyệt theo hàng

Không dùng được trực tiếp chiều cao "cùng hàng sắp" vì điều kiện sắp nằm **dọc theo hàng**. Ta vẫn có thể cố định **cặp hàng trên/dưới** `(x1, x2)`: một cột hợp lệ cho cặp này nếu mọi ô trong cột `≤ k`; hai cột kề hợp lệ "nối" được nếu mọi hàng không tăng giữa chúng. Đếm đoạn cột liên tiếp ⇒ `O(m² n)` — cách nhìn thứ hai để đối chiếu.

In [7]:
def count_row_pairs(g, k):
    m, n = len(g), len(g[0]); total = 0
    for x1 in range(m):
        colok = [True] * n
        linkok = [True] * (n - 1)
        for x2 in range(x1, m):
            for y in range(n):
                colok[y] = colok[y] and g[x2][y] <= k
            for y in range(n - 1):
                linkok[y] = linkok[y] and g[x2][y] >= g[x2][y + 1]
            run = 0
            for y in range(n):
                if not colok[y]:
                    run = 0
                elif y and run and linkok[y - 1]:
                    run += 1
                else:
                    run = 1
                total += run
    return total

for g, k, exp in examples:
    print(count_row_pairs(g, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
36 | kỳ vọng 36
1 | kỳ vọng 1


## 8. TRACE từng bước

Trace cách 3 trên `[[3,2,4],[1,2,1],[3,1,1]]`, `k = 3` (có ô `> k` và chỗ hàng tăng).

In [8]:
def trace(g, k):
    W = widths(g, k); m, n = len(g), len(g[0])
    print('W =', W)
    total = 0
    for j in range(n):
        col = [W[i][j] for i in range(m)]
        sm = sum_suffix_mins(col)
        total += sum(sm)
        print(f"cột {j}: độ rộng {col} -> số hình theo góc dưới-phải {sm}")
    print('tổng:', total, '| vét cạn:', count_brute(g, k))

trace([[3, 2, 4], [1, 2, 1], [3, 1, 1]], 3)

W = [[1, 2, 0], [1, 1, 2], [1, 2, 3]]
cột 0: độ rộng [1, 1, 1] -> số hình theo góc dưới-phải [1, 2, 3]
cột 1: độ rộng [2, 1, 2] -> số hình theo góc dưới-phải [2, 2, 4]
cột 2: độ rộng [0, 2, 3] -> số hình theo góc dưới-phải [0, 2, 5]
tổng: 21 | vét cạn: 21


## 9. Code nộp LeetCode

(Chữ ký suy đoán — xem ghi chú Premium.)

In [9]:
class Solution:
    def countSubmatrices(self, grid: List[List[int]], k: int) -> int:
        m, n = len(grid), len(grid[0])
        stacks = [[] for _ in range(n)]       # ngăn xếp (độ rộng, số hàng) cho mỗi cột
        cur = [0] * n
        total = 0
        for i in range(m):
            row = grid[i]
            w = 0
            for j in range(n):
                v = row[j]
                if v > k:
                    w = 0
                elif j and w and row[j - 1] >= v:
                    w += 1
                else:
                    w = 1
                st = stacks[j]
                cnt = 1
                c = cur[j]
                while st and st[-1][0] >= w:
                    pw, pc = st.pop()
                    c -= pw * pc
                    cnt += pc
                st.append((w, cnt))
                c += w * cnt
                cur[j] = c
                total += c
        return total

for g, k, exp in examples:
    got = Solution().countSubmatrices(g, k)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
36 OK
1 OK


Bản nộp tính `W` "trực tuyến" theo hàng và giữ một ngăn xếp cho **mỗi cột**, nên chỉ cần duyệt ma trận một lần theo hàng (thân thiện bộ nhớ đệm).

## 10. Vì sao thuật toán đúng

1. **Đoạn hàng hợp lệ:** đoạn cột `[y1, y2]` trên hàng `x` hợp lệ (mọi giá trị `≤ k`, không tăng) ⇔ `y2 − y1 + 1 ≤ W[x][y2]` — vì `W[x][y2]` là đoạn hợp lệ dài nhất kết thúc tại `y2` và mọi đoạn con kết thúc tại `y2` của một đoạn hợp lệ cũng hợp lệ.
2. **Ma trận con** `[x1..x2] × [y1..y2]` hợp lệ ⇔ mọi hàng hợp lệ trên `[y1, y2]` ⇔ độ rộng `≤ min_{x} W[x][y2]`.
3. **Đếm theo góc dưới-phải `(x2, y2)`:** `Σ_{x1 ≤ x2} min_{x ∈ [x1, x2]} W[x][y2]`. Ngăn xếp đơn điệu duy trì đúng tổng này (mỗi khối `(w, c)` biểu diễn `c` giá trị `x1` liên tiếp có cùng min `w`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi ma trận con | `O(m³ n³)` | `O(1)` |
| 2. Đi lên giữ min | `O(m² n)` | `O(mn)` |
| 3. Ngăn xếp đơn điệu | `O(mn)` | `O(mn)` (hoặc `O(m + n)`) |
| 4. Cặp hàng | `O(m² n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'quadratic': count_quadratic, 'stack': count_stack, 'row_pairs': count_row_pairs,
         'submit': lambda g, k: Solution().countSubmatrices(g, k)}
edge = [([[5]], 4, 0), ([[1, 2]], 5, 2), ([[2, 1]], 5, 3), ([[2], [1]], 5, 3), ([[3, 3], [3, 3]], 3, 9)]
for g, k, exp in examples + edge:
    assert count_brute(g, k) == exp, (g, k, count_brute(g, k))
    for name, f in impls.items():
        assert f(g, k) == exp, (name, g, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3359)
cnt = 0
for _ in range(800):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = [[rng.randint(1, 4) for _ in range(n)] for _ in range(m)]
    k = rng.randint(1, 4)
    exp = count_brute(g, k)
    for name, f in impls.items():
        assert f(g, k) == exp, (name, g, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'ma trận ngẫu nhiên.')

Khớp vét cạn trên 800 ma trận ngẫu nhiên.


### 12.3. Ca lớn `1000 × 1000`

In [12]:
import time
N = 1000
cases = {
    'giá trị 1..3, k = 3': ([[rng.randint(1, 3) for _ in range(N)] for _ in range(N)], 3),
    'toàn 1, k = 1': ([[1] * N for _ in range(N)], 1),
    'hàng giảm dần, k lớn': ([[N - j for j in range(N)] for _ in range(N)], 10 ** 9),
}
for name, (g, k) in cases.items():
    t = time.time(); a = Solution().countSubmatrices(g, k); t1 = time.time() - t
    print(f"{name:24s}: {a:16d} | {t1:.2f}s")
assert Solution().countSubmatrices([[1] * N for _ in range(N)], 1) == (N * (N + 1) // 2) ** 2
print('toàn 1 = (N(N+1)/2)^2: đúng')

giá trị 1..3, k = 3     :        503455090 | 0.43s


toàn 1, k = 1           :     250500250000 | 0.32s


hàng giảm dần, k lớn    :     250500250000 | 0.38s


toàn 1 = (N(N+1)/2)^2: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện "mỗi hàng không tăng" (chỉ kiểm `≤ k`)

In [13]:
def wrong_no_sorted(g, k):
    return count_stack([[1 if v <= k else 10 ** 9 for v in row] for row in g], 1)

print('sai :', wrong_no_sorted([[1, 2]], 5), '| đúng:', Solution().countSubmatrices([[1, 2]], 5))

sai : 3 | đúng: 2


**Vì sao sai:** `[1, 2]` là hàng **tăng** — chỉ hai ma trận con đơn lẻ hợp lệ.

### Lỗi 2 — Kiểm "không giảm" thay vì "không tăng"

In [14]:
def wrong_direction(g, k):
    return count_stack([row[::-1] for row in g], k)

print('sai :', wrong_direction([[2, 1]], 5), '| đúng:', Solution().countSubmatrices([[2, 1]], 5))

sai : 2 | đúng: 3


**Vì sao sai:** đảo chiều điều kiện cho đáp án của ma trận phản chiếu (`[[2,1]]` thành 2 thay vì 3).

### Lỗi 3 — Khi gặp ô `> k`, quên đặt độ rộng về 0 (chỉ đặt lại 1)

In [15]:
def wrong_reset(g, k):
    m, n = len(g), len(g[0])
    W = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if j and W[i][j - 1] and g[i][j - 1] >= g[i][j]:
                W[i][j] = W[i][j - 1] + 1
            else:
                W[i][j] = 1                                   # SAI: ô > k vẫn được tính
    total = 0
    for j in range(n):
        total += sum(sum_suffix_mins([W[i][j] for i in range(m)]))
    return total

print('sai :', wrong_reset(*examples[0][:2]), '| đúng:', Solution().countSubmatrices(*examples[0][:2]))

sai : 30 | đúng: 8


**Vì sao sai:** ô có giá trị `> k` không được thuộc bất kỳ ma trận con nào.

### Lỗi 4 — Ngăn xếp đơn điệu gộp khối với điều kiện `>` (để lại khối bằng nhau) mà không gộp số đếm đúng

In [16]:
def wrong_stack_strict(g, k):
    W = widths(g, k); m, n = len(g), len(g[0]); total = 0
    for j in range(n):
        st, cur = [], 0
        for i in range(m):
            w = W[i][j]
            while st and st[-1][0] > w:
                pw, pc = st.pop(); cur -= pw * pc
            st.append((w, 1)); cur += w                       # SAI: không cộng số hàng đã gộp
            total += cur
    return total

g4 = [[3, 1], [2, 3]]
print('sai :', wrong_stack_strict(g4, 3), '| đúng:', Solution().countSubmatrices(g4, 3), '| vét cạn:', count_brute(g4, 3))

sai : 6 | đúng: 7 | vét cạn: 7


**Vì sao sai:** khi bật các khối có độ rộng lớn hơn, số hàng của chúng không biến mất — chúng được "hạ" xuống độ rộng mới; phải cộng `cnt` của các khối bị gộp vào khối mới.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm ma trận con thỏa điều kiện theo hàng" ⇒ độ rộng hợp lệ kết thúc tại mỗi ô (giống "count submatrices with all ones", LC 1504).
- Tổng các min của mọi đoạn hậu tố ⇒ ngăn xếp đơn điệu với cặp `(giá trị, số lượng)`.
- Điều kiện thứ tự trong hàng ⇒ chỉ ảnh hưởng cách tính `W`.

## 15. Biến thể của bài

1. **LC 1504 — Count Submatrices With All Ones:** `W` = số ô 1 liên tiếp.
2. Điều kiện cả **hàng và cột** không tăng ⇒ thêm ràng buộc dọc — cần cách khác (ví dụ cặp hàng + kiểm tra cột).
3. Đếm ma trận con có max đúng bằng `k` ⇒ `f(k) − f(k − 1)`.
4. Tìm **diện tích lớn nhất** thay vì đếm ⇒ histogram lớn nhất (LC 85).

In [17]:
# Biến thể 1 (LC 1504) và biến thể 3
def ones_count(mat):
    m, n = len(mat), len(mat[0])
    W = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                W[i][j] = (W[i][j - 1] if j else 0) + 1
    return sum(sum(sum_suffix_mins([W[i][j] for i in range(m)])) for j in range(n))

assert ones_count([[1, 0, 1], [1, 1, 0], [1, 1, 0]]) == 13 and ones_count([[0, 1, 1, 0], [0, 1, 1, 1], [1, 1, 1, 0]]) == 24
def exactly_k(g, k):
    return Solution().countSubmatrices(g, k) - (Solution().countSubmatrices(g, k - 1) if k > 1 else 0)
rng = random.Random(15)
for _ in range(200):
    g = [[rng.randint(1, 4) for _ in range(rng.randint(1, 4))] for _ in range(rng.randint(1, 4))]
    g = [row[:len(g[0])] + [1] * (len(g[0]) - len(row)) for row in g]
    k = rng.randint(1, 4)
    exp = sum(1 for x1 in range(len(g)) for x2 in range(x1, len(g)) for y1 in range(len(g[0])) for y2 in range(y1, len(g[0]))
              if max(g[x][y] for x in range(x1, x2 + 1) for y in range(y1, y2 + 1)) == k
              and all(g[x][y] >= g[x][y + 1] for x in range(x1, x2 + 1) for y in range(y1, y2)))
    assert exactly_k(g, k) == exp
print('LC 1504 ví dụ: 13, 24 — đúng; biến thể "max đúng bằng k" khớp vét cạn.')

LC 1504 ví dụ: 13, 24 — đúng; biến thể "max đúng bằng k" khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi ma trận con | `O(m³n³)` | Oracle |
| 2 | Đi lên giữ min | `O(m²n)` | Đơn giản |
| 3 | Ngăn xếp đơn điệu | `O(mn)` | **Nộp bài** |
| 4 | Cặp hàng | `O(m²n)` | Góc nhìn khác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[3, 2, 1]]`, `k = 3`.

**Bài 2 (tính tay):** `[[2, 2], [1, 3]]`, `k = 2`.

**Bài 3 (điền TODO):** tìm **diện tích lớn nhất** của ma trận con hợp lệ.

**Bài 4 (tự viết lại):** biến thể 2 — thêm điều kiện mỗi **cột** cũng không tăng (từ trên xuống).

**Bài 5 (mini challenge):** đếm với điều kiện mỗi hàng **tăng nghiêm ngặt**.

In [18]:
def max_area_todo(g, k):
    # TODO
    raise NotImplementedError

def count_rows_and_cols_sorted_todo(g, k):
    # TODO
    raise NotImplementedError

def count_strictly_increasing_rows_todo(g, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một hàng, giảm dần. (2) Mọi đoạn con. (3) `6`.

**Bài 2.** (1) `W = [[1, 2], [1, 0]]`. (2) Cột 0: `1 + 2`; cột 1: `2 + 0`. (3) `5`.

**Bài 3.** (1) Với mỗi cột, histogram `W`. (2) Hình chữ nhật lớn nhất trong histogram (ngăn xếp). (3) Lấy max.

**Bài 4.** (1) Cố định cặp hàng; một cột hợp lệ nếu không tăng dọc. (2) Đếm đoạn cột liên tiếp. (3) Vét cạn kiểm tra.

**Bài 5.** (1) Đổi điều kiện nối `g[j−1] < g[j]`. (2) Phần còn lại giữ nguyên. (3) Vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().countSubmatrices([[3, 2, 1]], 3) == 6 and Solution().countSubmatrices([[2, 2], [1, 3]], 2) == 5 == count_brute([[2, 2], [1, 3]], 2)

def max_area(g, k):
    W = widths(g, k); m, n = len(g), len(g[0]); best = 0
    for j in range(n):
        h = [W[i][j] for i in range(m)] + [0]
        st = []
        for i, w in enumerate(h):
            start = i
            while st and st[-1][1] >= w:
                s0, sw = st.pop(); best = max(best, sw * (i - s0)); start = s0
            st.append((start, w))
    return best

def max_area_brute(g, k):
    m, n = len(g), len(g[0]); best = 0
    for x1 in range(m):
        for x2 in range(x1, m):
            for y1 in range(n):
                for y2 in range(y1, n):
                    if all(g[x][y] <= k for x in range(x1, x2 + 1) for y in range(y1, y2 + 1)) and \
                       all(g[x][y] >= g[x][y + 1] for x in range(x1, x2 + 1) for y in range(y1, y2)):
                        best = max(best, (x2 - x1 + 1) * (y2 - y1 + 1))
    return best

rng = random.Random(3)
for _ in range(300):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = [[rng.randint(1, 4) for _ in range(n)] for _ in range(m)]; k = rng.randint(1, 4)
    assert max_area(g, k) == max_area_brute(g, k)
print('Bài 1: 6 | Bài 2: 5 | Bài 3 khớp vét cạn.')

Bài 1: 6 | Bài 2: 5 | Bài 3 khớp vét cạn.


In [20]:
# Bài 4, 5
def count_rows_and_cols_sorted(g, k):
    m, n = len(g), len(g[0]); total = 0
    for x1 in range(m):
        colok = [True] * n; linkok = [True] * (n - 1); vert = [True] * n
        for x2 in range(x1, m):
            for y in range(n):
                colok[y] = colok[y] and g[x2][y] <= k
                if x2 > x1:
                    vert[y] = vert[y] and g[x2 - 1][y] >= g[x2][y]
            for y in range(n - 1):
                linkok[y] = linkok[y] and g[x2][y] >= g[x2][y + 1]
            run = 0
            for y in range(n):
                if not (colok[y] and vert[y]):
                    run = 0
                elif y and run and linkok[y - 1]:
                    run += 1
                else:
                    run = 1
                total += run
    return total

def count_strictly_increasing_rows(g, k):
    m, n = len(g), len(g[0])
    W = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if g[i][j] <= k:
                W[i][j] = W[i][j - 1] + 1 if j and W[i][j - 1] and g[i][j - 1] < g[i][j] else 1
    return sum(sum(sum_suffix_mins([W[i][j] for i in range(m)])) for j in range(n))

def brute_cond(g, k, rowok, colok_fn):
    m, n = len(g), len(g[0]); c = 0
    for x1 in range(m):
        for x2 in range(x1, m):
            for y1 in range(n):
                for y2 in range(y1, n):
                    if all(g[x][y] <= k for x in range(x1, x2 + 1) for y in range(y1, y2 + 1)) and \
                       all(rowok(g[x][y], g[x][y + 1]) for x in range(x1, x2 + 1) for y in range(y1, y2)) and \
                       all(colok_fn(g[x][y], g[x + 1][y]) for x in range(x1, x2) for y in range(y1, y2 + 1)):
                        c += 1
    return c

rng = random.Random(5)
for _ in range(300):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    g = [[rng.randint(1, 4) for _ in range(n)] for _ in range(m)]; k = rng.randint(1, 4)
    assert count_rows_and_cols_sorted(g, k) == brute_cond(g, k, lambda a, b: a >= b, lambda a, b: a >= b)
    assert count_strictly_increasing_rows(g, k) == brute_cond(g, k, lambda a, b: a < b, lambda a, b: True)
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
W[i][j] = 0 nếu g > k; W[i][j-1] + 1 nếu (j > 0, W[i][j-1] > 0, g[i][j-1] >= g[i][j]); ngược lại 1
cho mỗi cột j: duyệt i, ngăn xếp (w, cnt) tăng dần;
    gộp khối có w' >= w (cur -= w'·cnt'), đẩy (w, tổng cnt), cur += w·cnt; ans += cur
```

**Checklist:**

- [ ] Ô `> k` có `W = 0`?
- [ ] Điều kiện nối "không tăng": `g[j−1] ≥ g[j]`?
- [ ] Ngăn xếp gộp khối `≥` và cộng dồn số hàng?
- [ ] Kết quả có thể tới `~2.5·10^{11}` (C++/Java: `long long`)?
- [ ] Đối chiếu chữ ký hàm nếu có Premium?